# COMP8240 — feasibility check for the Min-K% Prob replication

Runs the original code from Shi et al. (ICLR 2024) on the original WikiMIA data,
end to end, so the proposal can honestly say the software was downloaded and executed.

**How to use:** open in Google Colab, `Runtime → Run all`. A GPU is *not* required —
Pythia-160M runs on the free CPU runtime in a few minutes. Selecting a T4 makes it faster.

The last cell prints the four numbers to paste into Section 4.5 of the proposal:
model, runtime, Min-20% Prob AUC, and PPL baseline AUC.

## 1. Clone the original repository

In [ ]:
!git clone --quiet https://github.com/swj0419/detect-pretrain-code.git
!cd detect-pretrain-code && git log -1 --format='commit %h  (%ad)' --date=short
!ls detect-pretrain-code/src

## 2. Pin the environment

The repository ships no `requirements.txt`, so we pin versions ourselves — this is
issue (i) noted in Section 4.2 of the proposal. `accelerate` is needed because
`run.py` loads models with `device_map='auto'`; `openai` is needed only because
`run.py` imports it unconditionally, even on the local-model path.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate>=0.33" "openai" "scikit-learn" "matplotlib" "tqdm"
import transformers, datasets, sklearn, torch
print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('sklearn     ', sklearn.__version__)

## 3. Check the original data downloads

WikiMIA is 469 KB in total. The four splits should report 776 / 542 / 250 / 82 examples.

In [ ]:
from datasets import load_dataset
from collections import Counter

for L in [32, 64, 128, 256]:
    d = load_dataset('swj0419/WikiMIA', split=f'WikiMIA_length{L}')
    print(f'length {L:>3}: n={len(d):>4}  labels={dict(Counter(d["label"]))}  columns={d.column_names}')

d32 = load_dataset('swj0419/WikiMIA', split='WikiMIA_length32')
print('\nexample non-member (label 0):\n ', d32[0]['input'][:200])

## 4. Run the original pipeline

`--target_model` is the model under attack; `--ref_model` is only used by the
*Smaller Ref* baseline. Swap in `EleutherAI/pythia-1.4b` or `EleutherAI/pythia-2.8b`
on a T4 once this smaller run works.

In [ ]:
TARGET = 'EleutherAI/pythia-160m'
REF    = 'EleutherAI/pythia-70m'
LENGTH = 32

import time, subprocess, os
start = time.time()
proc = subprocess.run(
    ['python', 'run.py', '--target_model', TARGET, '--ref_model', REF,
     '--data', 'swj0419/WikiMIA', '--length', str(LENGTH)],
    cwd='detect-pretrain-code/src', capture_output=True, text=True)
RUNTIME = time.time() - start
print(proc.stdout[-3000:])
print(proc.stderr[-2000:] if proc.returncode else '')
print(f'\nreturn code {proc.returncode}  |  wall clock {RUNTIME/60:.1f} min')

## 5. Read the results the code wrote

`eval.py` writes `auc.txt` and the ROC figure `auc.png` into its output directory.

In [ ]:
import glob, re

auc_files = glob.glob('detect-pretrain-code/src/out/**/auc.txt', recursive=True)
print('output dir:', os.path.dirname(auc_files[0]), '\n')
text = open(auc_files[0]).read()
print(text)

def auc_for(name):
    for line in text.strip().splitlines():
        if line.startswith(name):
            return float(re.search(r'AUC ([0-9.]+)', line).group(1))
    return None

mink20 = auc_for('Min_20.0% Prob')
ppl    = auc_for('ppl')

print('=' * 62)
print('PASTE THESE INTO SECTION 4.5 OF THE PROPOSAL')
print('=' * 62)
print(f'  model          : {TARGET}')
print(f'  runtime        : {RUNTIME/60:.1f} minutes')
print(f'  Min-20% AUC    : {mink20:.3f}')
print(f'  PPL baseline   : {ppl:.3f}')

In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(os.path.dirname(auc_files[0]), 'auc.png')))

## 6. Independent re-implementation (sanity check)

Thirty lines that recompute Min-K% Prob from scratch. If this agrees with the
repository's number, the sign convention noted as issue (iii) in Section 4.2 has
been read correctly; if it comes out at `1 - AUC`, the ROC curve is inverted.

In [ ]:
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

tok = AutoTokenizer.from_pretrained(TARGET)
model = AutoModelForCausalLM.from_pretrained(TARGET).eval()

def token_logprobs(text):
    ids = torch.tensor(tok.encode(text)).unsqueeze(0)
    with torch.no_grad():
        logits = model(ids).logits
    lp = torch.log_softmax(logits, dim=-1)[0]
    return [lp[i, t].item() for i, t in enumerate(ids[0][1:])]

def min_k_prob(text, k=0.2):
    lps = token_logprobs(text)
    n = max(1, int(len(lps) * k))
    return float(np.mean(sorted(lps)[:n]))   # higher (less negative) => member

scores = [min_k_prob(ex['input']) for ex in d32]
labels = [ex['label'] for ex in d32]
print(f'independent Min-20% AUC = {roc_auc_score(labels, scores):.3f}')
print(f'repository  Min-20% AUC = {mink20:.3f}')